In [1]:
print("helo")

helo


In [2]:
import os
import glob
import pandas as pd

In [30]:
def clean_all_amazon_datasets(input_folder, output_folder):
    """
    Loop through all CSV files in input_folder, clean them using all cleaning functions,
    and save cleaned files in output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    for file in csv_files:
        print(f"Cleaning dataset: {file}")
        df = pd.read_csv(file)
        
        # --- Apply all cleaning functions sequentially ---
        # df = clean_order_date(df)
        # df = clean_original_price(df)
        # df = clean_customer_rating(df)
        # df = clean_customer_city(df)
        # df = clean_boolean_columns(df)
        # df = clean_product_category(df)
        # df = clean_delivery_days(df)
        # df = handle_duplicates(df)
        # df = handle_outlier_prices(df)
        # df = clean_payment_methods(df)
        
        # Save cleaned dataset
        base_name = os.path.basename(file)
        output_path = os.path.join(output_folder, base_name)
        df.to_csv(output_path, index=False)
        print(f"Saved cleaned dataset: {output_path}")

In [57]:
import pandas as pd
from datetime import datetime

def clean_order_date(df, date_column='order_date'):

    def parse_date(x):
        if pd.isna(x):
            return pd.NaT
        x = str(x).strip()  # remove whitespace
        # Try multiple known formats
        for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%d-%m-%y", "%Y-%m-%d", "%Y/%m/%d"):
            try:
                return pd.to_datetime(datetime.strptime(x, fmt))
            except:
                continue
        # If none matches, return NaT
        return pd.NaT

    df[date_column] = df[date_column].apply(parse_date)
    
    # Standardize format to YYYY-MM-DD
    df[date_column] = df[date_column].dt.strftime('%Y-%m-%d')
    
    return df


In [58]:
df_2015 = pd.read_csv("datasets/amazon_india_2015.csv")
# df_2015['order_date'].sample(10)
df_2015 = clean_order_date(df_2015, 'order_date')
print(df_2015['order_date'].sample(10))

8464     2015-04-29
14164    2015-06-26
5683     2015-03-11
11480    2015-05-31
12867    2015-06-07
8633     2015-04-11
8303     2015-04-27
4630     2015-03-02
31707    2015-12-30
6582     2015-03-11
Name: order_date, dtype: object


In [59]:
import re

def clean_original_price(df, price_column='original_price_inr'):

    def parse_price(x):
        if pd.isna(x):
            return pd.NA
        x = str(x).strip()  # remove spaces
        # Remove ₹ and commas
        x = re.sub(r'[₹,]', '', x)
        # Check if numeric
        if x.replace('.', '', 1).isdigit():
            return float(x)
        else:
            return pd.NA
    
    df[price_column] = df[price_column].apply(parse_price)
    
    return df


In [61]:
df_2015 = pd.read_csv("datasets/amazon_india_2015.csv")
# df_2015['original_price_inr'].head(10)
df_2015 = clean_original_price(df_2015, 'original_price_inr')
print(df_2015['original_price_inr'].unique()[:10])


[123614.29 54731.86 97644.25 21947.26 131194.65 86987.64 32169.01 40264.16
 88664.85 73967.02]


In [62]:
import re


def clean_customer_rating(df, rating_column='customer_rating'):

    def parse_rating(x):
        if pd.isna(x):
            return pd.NA
        x = str(x).strip()
        
        # Extract first numeric pattern (e.g., 4, 4.5)
        match = re.search(r'(\d+(\.\d+)?)', x)
        if match:
            val = float(match.group(1))
            # Keep only valid 0–5 ratings
            if 0 <= val <= 5:
                return val
        return pd.NA
    
    df[rating_column] = df[rating_column].apply(parse_rating)
    return df


In [63]:
# Load your dataset
df = pd.read_csv("datasets/amazon_india_2015.csv")

# Apply the cleaning function
df = clean_customer_rating(df, 'customer_rating')

# Check the results
print(df['customer_rating'].unique()[:10])


[5.0 4.5 <NA> 3.0 4.0 3.5]


In [84]:
import pandas as pd
import re

def clean_customer_city(df):
    """
    Dynamically standardize city names in the 'customer_city' column.
    Handles variations, slashes, casing, and spelling errors.
    """
    if 'customer_city' not in df.columns:
        print("Warning: 'customer_city' column not found.")
        return df

    def normalize_city_name(city):
        if pd.isna(city):
            return pd.NA
        
        # Lowercase for uniformity
        city = str(city).strip().lower()

        # Replace slashes, hyphens, extra spaces
        city = re.sub(r"[/\-]", " ", city)
        city = re.sub(r"\s+", " ", city)

        # Correct common variations dynamically
        replacements = {
            'bangalore': 'bengaluru',
            'bombay': 'mumbai',
            'delhi': 'new delhi',
            'madras': 'chennai',
            'calcutta': 'kolkata',
        }
        for old, new in replacements.items():
            if old in city:
                city = new

        # Capitalize first letter of each word
        return city.title()

    # Apply cleaning
    df['customer_city'] = df['customer_city'].apply(normalize_city_name)

    # Replace empty or invalid strings with pd.NA
    df['customer_city'] = df['customer_city'].replace(["", "na", "none", "null"], pd.NA)

    return df


In [104]:
df_2015 = pd.read_csv("datasets/amazon_india_2015.csv")
# Clean the customer_city column
df_2015 = clean_customer_city(df_2015)

# Preview cleaned data
print(df_2015[['customer_city']].sample(10))

      customer_city
18335        Jaipur
13080       Kolkata
25530        Mumbai
13320       Chennai
17655        Nagpur
11274          Pune
23622       Lucknow
10439     New Delhi
2266        Chennai
6660        Chennai


In [105]:
num_missing = df_2015['customer_city'].isna().sum()
num_missing

np.int64(0)

In [123]:
def clean_boolean_columns(df, bool_columns=None):
    """
    Standardize boolean columns to True/False and use pd.NA for missing/invalid entries.
    """
    if bool_columns is None:
        # Default columns to clean
        bool_columns = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']
    
    def to_boolean(x):
        if pd.isna(x):
            return pd.NA
        x_str = str(x).strip().lower()
        if x_str in ['true', '1', 'yes', 'y']:
            return True
        elif x_str in ['false', '0', 'no', 'n']:
            return False
        else:
            return pd.NA  # Anything else treated as missing

    for col in bool_columns:
        if col in df.columns:
            df[col] = df[col].apply(to_boolean)
    
    return df


In [156]:
import pandas as pd
df_2015 = pd.read_csv("datasets/amazon_india_2015.csv")
df_2015 = clean_boolean_columns(df_2015)
print(df_2015[['is_prime_member', 'is_prime_eligible', 'is_festival_sale']].sample(10))


       is_prime_member  is_prime_eligible  is_festival_sale
22967            False               True             False
4532             False              False              True
13934            False              False              True
28743            False               True             False
21464            False               True              True
19516            False              False             False
32393            False              False             False
29024            False               True             False
8563             False               True             False
2181             False               True             False


In [157]:
import pandas as pd

def clean_product_category(df, category_column='category'):
    """
    Standardize product category names:
    - Handles variations like 'Electronics', 'Electronic', 'ELECTRONICS', 'Electronicss'
    - Keeps consistent naming as either 'Electronics' or 'Electronics & Accessories'
    - Replaces missing or invalid values with pd.NA
    """

    def standardize_category(cat):
        if pd.isna(cat):
            return pd.NA
        cat = str(cat).strip().lower()

        if 'accessor' in cat:
            return 'Electronics & Accessories'
        elif cat in ['electronics', 'electronic', 'electronicss', 'electronics']:
            return 'Electronics'
        else:
            return cat.title()

    df[category_column] = df[category_column].apply(standardize_category)
    return df


In [158]:
df = pd.read_csv("datasets/amazon_india_2020.csv")
df = clean_product_category(df, 'category')
print(df['category'].unique()[:10])


['Electronics' 'Electronics & Accessories']


In [159]:
import pandas as pd
import re

def clean_delivery_days(df, column_name='delivery_days'):

    def parse_delivery(value):
        if pd.isna(value):
            return pd.NA

        val = str(value).strip().lower()

        # Handle common text formats
        if val in ['same day', 'express']:
            return 0
        elif re.match(r'^\d+\s*-\s*\d+', val):        # '1-2 days'
            nums = [int(x) for x in re.findall(r'\d+', val)]
            return round(sum(nums) / len(nums))
        elif re.search(r'\d+', val):                  # '5' or '3 days'
            num = int(re.search(r'\d+', val).group())
            return num if 0 <= num <= 30 else pd.NA
        else:
            return pd.NA

    df[column_name] = df[column_name].apply(parse_delivery).astype('Int64')
    return df


In [160]:
df = clean_delivery_days(df)
print(df['delivery_days'].unique()[:30])

<IntegerArray>
[1, 4, 5, 3, 6, 15, 2, 0, 7]
Length: 9, dtype: Int64


In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143715 entries, 0 to 143714
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          143715 non-null  object 
 1   order_date              143715 non-null  object 
 2   customer_id             143715 non-null  object 
 3   product_id              143715 non-null  object 
 4   product_name            143715 non-null  object 
 5   category                143715 non-null  object 
 6   subcategory             143715 non-null  object 
 7   brand                   143715 non-null  object 
 8   original_price_inr      143715 non-null  object 
 9   discount_percent        143715 non-null  float64
 10  discounted_price_inr    143715 non-null  float64
 11  quantity                143715 non-null  int64  
 12  subtotal_inr            143715 non-null  float64
 13  delivery_charges        132197 non-null  float64
 14  final_amount_inr    

In [162]:
def handle_duplicates(df):
    key_cols =['customer_id', 'order_date', 'product_id','final_amount_inr']
    dup_count = df.duplicated(subset=key_cols,keep =False)
    duplicates  = df[dup_count]
    def resolve_duplicates(group):
        if group['quantity'].sum() > group['quantity'].iloc[0]:
            return group
        else:
            return group.iloc[0]
    df_cleaned = duplicates.groupby(key_cols,group_keys = False).apply(resolve_duplicates)
    df_non_duplicates = df[~dup_count]
    df_final = pd.concat([df_cleaned, df_non_duplicates], ignore_index=True)
    return df_final

In [164]:
import pandas as pd
import numpy as np

def handle_outlier_prices_domain1(df, price_col='original_price_inr', category_col='category', factor_high=100, factor_low=0.01):

    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
    
    # Calculate median price per category
    category_median = df.groupby(category_col)[price_col].transform('median')
    
    # Replace unrealistic high prices
    df[price_col] = np.where(df[price_col] > factor_high * category_median, category_median, df[price_col])
    
    # Replace unrealistic low prices
    df[price_col] = np.where(df[price_col] < factor_low * category_median, category_median, df[price_col])
    
    return df



In [186]:
df = pd.read_csv("datasets/amazon_india_2016.csv")
print(df['original_price_inr'][915])
df = handle_outlier_prices_domain1(df)
df.info()

7529867.999999999
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55275 entries, 0 to 55274
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          55275 non-null  object 
 1   order_date              55275 non-null  object 
 2   customer_id             55275 non-null  object 
 3   product_id              55275 non-null  object 
 4   product_name            55275 non-null  object 
 5   category                55275 non-null  object 
 6   subcategory             55275 non-null  object 
 7   brand                   55275 non-null  object 
 8   original_price_inr      50287 non-null  float64
 9   discount_percent        55275 non-null  float64
 10  discounted_price_inr    55275 non-null  float64
 11  quantity                55275 non-null  int64  
 12  subtotal_inr            55275 non-null  float64
 13  delivery_charges        50851 non-null  float64
 14  final_amount_inr    

In [166]:
import pandas as pd
import numpy as np

def handle_outlier_prices_domain(df, price_col='original_price_inr', category_col='category', factor_high=100, factor_low=0.01):
    """
    Identify and correct unrealistic price outliers using per-category median-based domain logic.
    Also prints summary of outliers before correction.
    """
    # Ensure both required columns exist
    if price_col not in df.columns or category_col not in df.columns:
        print(f"❌ Missing required columns: {price_col} or {category_col}")
        return df

    # Step 1: Convert price column to numeric safely
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')

    # Step 2: Compute per-category median
    category_median = df.groupby(category_col)[price_col].transform('median')

    # Handle cases where median could be NaN
    category_median = category_median.fillna(df[price_col].median())

    # Step 3: Detect outliers before correction
    high_mask = df[price_col] > factor_high * category_median
    low_mask = df[price_col] < factor_low * category_median

    high_outliers = df[high_mask]
    low_outliers = df[low_mask]

    print(f"High outliers detected: {len(high_outliers)}")
    print(f"Low outliers detected: {len(low_outliers)}")

    if not high_outliers.empty:
        print("\nSample high outliers:")
        print(high_outliers[[category_col, price_col]].head())

    if not low_outliers.empty:
        print("\nSample low outliers:")
        print(low_outliers[[category_col, price_col]].head())

    # Step 4: Replace unrealistic values with category median
    df.loc[high_mask, price_col] = category_median[high_mask]
    df.loc[low_mask, price_col] = category_median[low_mask]

    print("\n✅ Outlier correction complete.\n")
    return df


In [167]:
df = pd.read_csv("datasets/amazon_india_2016.csv")
df = handle_outlier_prices_domain(df)

High outliers detected: 71
Low outliers detected: 132

Sample high outliers:
         category  original_price_inr
915   Electronics           7529868.0
2597  Electronics           9976461.0
3103  Electronics          13211301.0
4446  Electronics           5978212.0
4882  Electronics          16573738.0

Sample low outliers:
         category  original_price_inr
134   Electronics           -47502.48
613   Electronics           -55739.57
706   Electronics           -44939.24
753   Electronics           -75298.68
1072  Electronics           -10106.28

✅ Outlier correction complete.



In [168]:
import re

def clean_payment_methods(df, column='payment_method'):
    """
    Standardize payment method names and ensure consistent categorical naming.
    """
    if column not in df.columns:
        print(f"❌ Column '{column}' not found in DataFrame.")
        return df

    # Convert to lowercase and strip spaces
    df[column] = df[column].astype(str).str.lower().str.strip()

    # Define mapping patterns
    payment_mapping = {
        r'\bupi\b|\bphonepe\b|\bgoogle\s?pay\b|\bpaytm\b|\bbhim\b': 'UPI',
        r'\bcredit\s?card\b|\bcreditcard\b|\bcc\b|\bmastercard\b|\bvisa\b': 'Credit Card',
        r'\bdebit\s?card\b|\bdebitcard\b': 'Debit Card',
        r'\bcash\s?on\s?delivery\b|\bcod\b|\bc\.o\.d\b': 'Cash on Delivery',
        r'\bnet\s?banking\b|\bonline\s?banking\b': 'Net Banking',
        r'\bwallet\b': 'Wallet',
        r'\bemi\b': 'EMI',
    }

    # Apply replacements
    for pattern, replacement in payment_mapping.items():
        df[column] = df[column].apply(lambda x: replacement if re.search(pattern, x) else x)

    # Convert to title case for consistency
    df[column] = df[column].str.title()

    # Optional: print summary
    print("\n✅ Payment method cleaning complete. Unique values after cleaning:")
    print(df[column].unique())

    return df


In [169]:
print(df['payment_method'].unique()[:10])
df = clean_payment_methods(df)
print(df['payment_method'].unique()[:10])


['COD' 'Credit Card' 'Debit Card' 'Net Banking' 'UPI']

✅ Payment method cleaning complete. Unique values after cleaning:
['Cash On Delivery' 'Credit Card' 'Debit Card' 'Net Banking' 'Upi']
['Cash On Delivery' 'Credit Card' 'Debit Card' 'Net Banking' 'Upi']


In [170]:
def clean_all_amazon_datasets(input_folder, output_folder):
    """
    Loop through all CSV files in input_folder, clean them using all cleaning functions,
    and save cleaned files in output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    for file in csv_files:
        print(f"Cleaning dataset: {file}")
        df = pd.read_csv(file)
        
        # --- Apply all cleaning functions sequentially ---
        df = clean_order_date(df)
        df = clean_original_price(df)
        df = clean_customer_rating(df)
        df = clean_customer_city(df)
        df = clean_boolean_columns(df)
        df = clean_product_category(df)
        df = clean_delivery_days(df)
        df = handle_duplicates(df)
        df = handle_outlier_prices_domain1(df)
        df = clean_payment_methods(df)
        
        # Save cleaned dataset
        base_name = os.path.basename(file)
        output_path = os.path.join(output_folder, base_name)
        df.to_csv(output_path, index=False)
        print(f"Saved cleaned dataset: {output_path}")

In [171]:
clean_all_amazon_datasets("datasets/", "cleaned_datasets/")

Cleaning dataset: datasets\amazon_india_2015.csv


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\3623497702.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = duplicates.groupby(key_cols,group_keys = False).apply(resolve_duplicates)



✅ Payment method cleaning complete. Unique values after cleaning:
['Cash On Delivery' 'Credit Card' 'Debit Card' 'Net Banking']
Saved cleaned dataset: cleaned_datasets/amazon_india_2015.csv
Cleaning dataset: datasets\amazon_india_2016.csv


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\3623497702.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = duplicates.groupby(key_cols,group_keys = False).apply(resolve_duplicates)



✅ Payment method cleaning complete. Unique values after cleaning:
['Net Banking' 'Cash On Delivery' 'Debit Card' 'Credit Card' 'Upi']
Saved cleaned dataset: cleaned_datasets/amazon_india_2016.csv
Cleaning dataset: datasets\amazon_india_2017.csv


...


✅ Payment method cleaning complete. Unique values after cleaning:
['Upi' 'Credit Card' 'Cash On Delivery' 'Debit Card' 'Bnpl' 'Net Banking'
 'Wallet']
Saved cleaned dataset: cleaned_datasets/amazon_india_2023.csv
Cleaning dataset: datasets\amazon_india_2024.csv


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\3623497702.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = duplicates.groupby(key_cols,group_keys = False).apply(resolve_duplicates)



✅ Payment method cleaning complete. Unique values after cleaning:
['Upi' 'Cash On Delivery' 'Credit Card' 'Debit Card' 'Bnpl' 'Wallet'
 'Net Banking']
Saved cleaned dataset: cleaned_datasets/amazon_india_2024.csv
Cleaning dataset: datasets\amazon_india_2025.csv


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\3623497702.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = duplicates.groupby(key_cols,group_keys = False).apply(resolve_duplicates)



✅ Payment method cleaning complete. Unique values after cleaning:
['Upi' 'Wallet' 'Cash On Delivery' 'Debit Card' 'Bnpl' 'Credit Card'
 'Net Banking']
Saved cleaned dataset: cleaned_datasets/amazon_india_2025.csv


In [173]:
print(df.nunique())

transaction_id            55275
order_date                 1304
customer_id               26273
product_id                  312
product_name                312
category                      5
subcategory                   6
brand                        24
original_price_inr          467
discount_percent           6319
discounted_price_inr      30662
quantity                      3
subtotal_inr              31275
delivery_charges              1
final_amount_inr          31275
customer_city                50
customer_state               15
customer_tier                 4
customer_spending_tier        3
customer_age_group            5
payment_method                5
delivery_days                13
delivery_type                 3
is_prime_member               8
is_festival_sale              8
festival_name                 8
customer_rating              21
return_status                 3
order_month                  12
order_year                    1
order_quarter                 4
product_

In [183]:
df = pd.read_csv("cleaned_datasets/amazon_india_2025.csv")
# print(df.nunique())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77373 entries, 0 to 77372
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          77373 non-null  object 
 1   order_date              76273 non-null  object 
 2   customer_id             77373 non-null  object 
 3   product_id              77373 non-null  object 
 4   product_name            77373 non-null  object 
 5   category                77373 non-null  object 
 6   subcategory             77373 non-null  object 
 7   brand                   77373 non-null  object 
 8   original_price_inr      74907 non-null  float64
 9   discount_percent        77373 non-null  float64
 10  discounted_price_inr    77373 non-null  float64
 11  quantity                77373 non-null  int64  
 12  subtotal_inr            77373 non-null  float64
 13  delivery_charges        71179 non-null  float64
 14  final_amount_inr        77373 non-null

In [184]:
df = pd.read_csv("datasets/amazon_india_2025.csv")

# print(df.nunique())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77385 entries, 0 to 77384
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          77385 non-null  object 
 1   order_date              77385 non-null  object 
 2   customer_id             77385 non-null  object 
 3   product_id              77385 non-null  object 
 4   product_name            77385 non-null  object 
 5   category                77385 non-null  object 
 6   subcategory             77385 non-null  object 
 7   brand                   77385 non-null  object 
 8   original_price_inr      77385 non-null  object 
 9   discount_percent        77385 non-null  float64
 10  discounted_price_inr    77385 non-null  float64
 11  quantity                77385 non-null  int64  
 12  subtotal_inr            77385 non-null  float64
 13  delivery_charges        71189 non-null  float64
 14  final_amount_inr        77385 non-null

In [194]:
import pandas as pd
import glob
import os

raw_folder = "datasets"
cleaned_folder = "cleaned_datasets"

for raw_file in glob.glob(os.path.join(raw_folder, "*.csv")):
    base_name = os.path.basename(raw_file)
    clean_file = os.path.join(cleaned_folder, base_name)
    
    df_raw = pd.read_csv(raw_file)
    df_clean = pd.read_csv(clean_file)
    
    print(f"\n--- Comparing: {base_name} ---")
    
    # Number of rows and columns
    print(f"Raw shape: {df_raw.shape}, Cleaned shape: {df_clean.shape}")
    
    # Missing values
    print(f"Missing values in raw: {df_raw.isna().sum().sum()}")
    print(f"Missing values in cleaned: {df_clean.isna().sum().sum()}")


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\557039568.py:13: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(clean_file)



--- Comparing: amazon_india_2015.csv ---
Raw shape: (33165, 34), Cleaned shape: (33161, 34)
Missing values in raw: 39180
Missing values in cleaned: 40736

--- Comparing: amazon_india_2016.csv ---
Raw shape: (55275, 34), Cleaned shape: (55267, 34)
Missing values in raw: 65633
Missing values in cleaned: 68173

--- Comparing: amazon_india_2017.csv ---
Raw shape: (77385, 34), Cleaned shape: (77375, 34)
Missing values in raw: 91885
Missing values in cleaned: 95480


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\557039568.py:13: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(clean_file)



--- Comparing: amazon_india_2018.csv ---
Raw shape: (99495, 34), Cleaned shape: (99479, 34)
Missing values in raw: 118044
Missing values in cleaned: 122643


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\557039568.py:13: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(clean_file)


...


--- Comparing: amazon_india_2022.csv ---
Raw shape: (132660, 34), Cleaned shape: (132638, 34)
Missing values in raw: 158607
Missing values in cleaned: 164775


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\557039568.py:13: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(clean_file)



--- Comparing: amazon_india_2023.csv ---
Raw shape: (127132, 34), Cleaned shape: (127120, 34)
Missing values in raw: 152144
Missing values in cleaned: 158183


C:\Users\Tasneem Inayath\AppData\Local\Temp\ipykernel_12660\557039568.py:13: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(clean_file)



--- Comparing: amazon_india_2024.csv ---
Raw shape: (121605, 34), Cleaned shape: (121585, 34)
Missing values in raw: 145119
Missing values in cleaned: 150791

--- Comparing: amazon_india_2025.csv ---
Raw shape: (77385, 34), Cleaned shape: (77373, 34)
Missing values in raw: 92706
Missing values in cleaned: 96260


In [5]:
import pandas as pd
import os

def create_and_save_star_schema_tables(df_master, df_catalog, save_path="star_schema_tables"):
    os.makedirs(save_path, exist_ok=True)

    # --- 1️⃣ Transactions Table ---
    transactions_cols = [
        "transaction_id", "order_date", "customer_id", "product_id",
        "quantity", "discount_percent", "discounted_price_inr",
        "subtotal_inr", "delivery_charges", "final_amount_inr",
        "payment_method", "delivery_days", "delivery_type",
        "is_prime_member", "is_festival_sale", "festival_name",
        "return_status", "order_month", "order_year", "order_quarter"
    ]
    transactions = df_master[transactions_cols].copy()
    transactions["order_date"] = pd.to_datetime(transactions["order_date"], errors="coerce")
    transactions.to_csv(os.path.join(save_path, "transactions.csv"), index=False)

    # --- 2️⃣ Products Table ---
    products_cols = [
        "product_id", "product_name", "category", "subcategory", "brand",
        "original_price_inr", "discounted_price_inr", "product_weight_kg", "product_rating"
    ]
    products = df_master[products_cols].drop_duplicates(subset=["product_id"]).copy()
    
    # Merge with catalog if extra hierarchy info exists
    if not df_catalog.empty:
        products = products.merge(
            df_catalog.drop_duplicates(subset=["product_id"]),
            on="product_id", how="left"
        )
    products.to_csv(os.path.join(save_path, "products.csv"), index=False)

    # --- 3️⃣ Customers Table ---
    customers_cols = [
        "customer_id", "customer_city", "customer_state", "customer_tier",
        "customer_spending_tier", "customer_age_group", "is_prime_member"
    ]
    customers = df_master[customers_cols].drop_duplicates(subset=["customer_id"]).copy()
    customers.to_csv(os.path.join(save_path, "customers.csv"), index=False)

    # --- 4️⃣ Time Dimension Table ---
    time_dim = (
        df_master[["order_date", "order_month", "order_year", "order_quarter"]]
        .drop_duplicates()
        .copy()
    )
    time_dim["order_date"] = pd.to_datetime(time_dim["order_date"], errors="coerce")
    time_dim["day"] = time_dim["order_date"].dt.day
    time_dim["month_name"] = time_dim["order_date"].dt.month_name()
    time_dim["week"] = time_dim["order_date"].dt.isocalendar().week
    time_dim["day_of_week"] = time_dim["order_date"].dt.day_name()
    time_dim.to_csv(os.path.join(save_path, "time_dimension.csv"), index=False)

    print("✅ Star schema tables created and saved successfully!")
    print(f"📂 Saved in: {os.path.abspath(save_path)}")


In [6]:
import pandas as pd

# Load your datasets first
df_master = pd.read_csv(r"C:\Projects\amazonanalytics\cleaned_datasets\amazon_master_2015_2025.csv")
df_catalog = pd.read_csv(r"C:\Projects\amazonanalytics\cleaned_datasets\amazon_india_products_catalog.csv")

# Now call the function
create_and_save_star_schema_tables(df_master, df_catalog)


✅ Star schema tables created and saved successfully!
📂 Saved in: c:\Projects\amazonanalytics\star_schema_tables


In [4]:
import os

# Path to your eda_results folder
eda_folder = r"C:\Projects\amazonanalytics\EDA\eda_results"

# Check if the folder exists
if os.path.exists(eda_folder):
    # Loop through each item in the folder
    for file in os.listdir(eda_folder):
        full_path = os.path.join(eda_folder, file)
        # Print only if it's a file (not a folder)
        if os.path.isfile(full_path):
            print(file)
else:
    print("The folder path does not exist.")

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Projects\amazonanalytics\star_schema_tables\time_dimension.csv")

print(df.)

['order_date', 'order_month', 'order_year', 'order_quarter', 'day', 'month_name', 'week', 'day_of_week']


In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Projects\amazonanalytics\star_schema_tables\customers.csv")

print(df["is_prime_member"].unique())

[False  True]


In [7]:
import pandas as pd

# Load your master dataset
df_master = pd.read_csv(r"C:\Projects\amazonanalytics\cleaned_datasets\amazon_master_2015_2025.csv")  # Replace with your actual file path

# Select only customer_id and is_prime_member columns
df_customers = df_master[['customer_id', 'is_prime_member']]

# Optional: remove duplicates if you want unique customers
df_customers_unique = df_customers.drop_duplicates()

# Save to a new CSV (optional)
df_customers_unique.to_csv(r"C:\Projects\amazonanalytics\star_schema_tables\customers_prime_status.csv", index=False)

# Display the first few rows
print(df_customers_unique["is_prime_member"].unique())


[False  True]


In [8]:
import os

eda_folder = r"C:\Projects\amazonanalytics\EDA\eda_results"

# Walk through all subdirectories and files
for root, dirs, files in os.walk(eda_folder):
    for file in files:
        print(os.path.join(root, file))

C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\shopping_frequency_by_age.csv
C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\shopping_frequency_by_age.png
C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\spending_by_age.csv
C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\spending_by_age.png
C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\subcategory_preferences_by_age.csv
C:\Projects\amazonanalytics\EDA\eda_results\age_group_analysis\subcategory_preferences_by_age.png
C:\Projects\amazonanalytics\EDA\eda_results\brand_market_analysis\brand_marketshare_Audio.png
C:\Projects\amazonanalytics\EDA\eda_results\brand_market_analysis\brand_marketshare_Laptops.png
C:\Projects\amazonanalytics\EDA\eda_results\brand_market_analysis\brand_marketshare_Smart Watch.png
C:\Projects\amazonanalytics\EDA\eda_results\brand_market_analysis\brand_marketshare_Smartphones.png
C:\Projects\amazonanalytics\EDA\eda_results\brand_market_analy